# Price Comparison Tool

## Overview
This notebook implements a price comparison tool that scrapes data from **Amazon** and **Flipkart** for a given laptop model (e.g., HP Pavilion). It extracts the Name, Price, and Rating, compares them, and visualizes the results.

## Requirements
- `selenium`
- `beautifulsoup4`
- `pandas`
- `matplotlib`
- `seaborn`
- Google Chrome installed

In [2]:

!pip install selenium beautifulsoup3 pandas matplotlib seaborn

ERROR: Could not find a version that satisfies the requirement beautifulsoup3 (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for beautifulsoup3


In [3]:
import time
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

#  1. Setup Chrome Options 
def get_driver_options():
    options = Options()
    # Standard options to avoid detection and errors
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    # Use a real user-agent
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    # Note: Headless mode can sometimes trigger bot detection on Flipkart. 
    # We'll run with head (visible browser) by default for better success rates.
    # options.add_argument("--headless") 
    return options

In [4]:
# 2. Amazon Scraper 
def scrape_amazon(query):
    print(f"Processing Amazon for: {query}...")
    options = get_driver_options()
    driver = webdriver.Chrome(options=options)
    
    results = []
    try:
        url = f"https://www.amazon.in/s?k={query.replace(' ', '+')}"
        driver.get(url)
        time.sleep(3)  # Wait for page load
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Amazon search result items selector
        items = soup.select('div[data-component-type="s-search-result"]')[:10]
        
        for item in items:
            try:
                # Name
                name_el = item.select_one('h2 a span') or item.select_one('h2')
                name = name_el.text.strip() if name_el else "Unknown"
                
                # Price
                price_el = item.select_one('.a-price-whole')
                if not price_el:
                    continue 
                price = float(price_el.text.replace(',', '').strip())
                
                # Rating
                rating_el = item.select_one('.a-icon-alt')
                rating = rating_el.text.split(' ')[0] if rating_el else "0"
                try:
                    rating = float(rating)
                except:
                    rating = 0.0
                
                results.append({
                    "Name": name,
                    "Price": price,
                    "Rating": rating,
                    "Source": "Amazon"
                })
            except Exception as e:
                continue
                
    except Exception as e:
        print(f"Amazon Error: {e}")
    finally:
        driver.quit()
        
    return results

In [5]:
# 3. Flipkart Scraper 
def scrape_flipkart(query):
    print(f"Processing Flipkart for: {query}...")
    options = get_driver_options()
    driver = webdriver.Chrome(options=options)
    
    results = []
    try:
        url = f"https://www.flipkart.com/search?q={query.replace(' ', '%20')}"
        driver.get(url)
        time.sleep(4)  
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Flipkart Selectors (Updated for 2024/2025 Layouts)
        # We check multiple common container classes
        items = soup.select('div.tUxRFH') or soup.select('div._1AtVbE.col-12-12') or soup.select('div._75nlfW')
        
        # Debug info
        if not items:
             print(f"Warning: No items found on Flipkart. Page Title: {driver.title}")
        
        items = items[:10]
        
        for item in items:
            try:
                # Name (Try multiple selectors)
                name_el = item.select_one('div.KzDlHZ') or item.select_one('div._4rR01T') or item.select_one('a.s1Q9rs') or item.select_one('.wjcEIp')
                if not name_el:
                    continue
                name = name_el.text.strip()
                
                # Price 
                price_el = item.select_one('div.Nx9bqj') or item.select_one('div._30jeq3')
                if not price_el:
                    continue
                price = float(price_el.text.replace('₹', '').replace(',', '').strip())
                
                # Rating
                rating_el = item.select_one('div.XQDdHH') or item.select_one('div._3LWZlK')
                rating = float(rating_el.text.strip()) if rating_el else 0.0
                
                results.append({
                    "Name": name,
                    "Price": price,
                    "Rating": rating,
                    "Source": "Flipkart"
                })
            except Exception as e:
                continue
                
    except Exception as e:
        print(f"Flipkart Error: {e}")
    finally:
        driver.quit()
        
    return results

In [6]:
# 4. Main Execution & Comparison 
QUERY = "HP Pavilion Laptop"

print("Starting Scrapers...")
amazon_data = scrape_amazon(QUERY)
print(f"Amazon Extracted: {len(amazon_data)} items")

flipkart_data = scrape_flipkart(QUERY)
print(f"Flipkart Extracted: {len(flipkart_data)} items")

# Combine Results
all_data = amazon_data + flipkart_data

if all_data:
    df = pd.DataFrame(all_data)
    
    # Export to CSV
    filename = "laptop_price_comparison.csv"
    df.to_csv(filename, index=False)
    print(f"\nSuccess! Data exported to {filename}")
    
    # Display Summary
    print("\n--- Top 5 Results ---")
    print(df.head())
    
    print("\n--- Average Price by Source ---")
    print(df.groupby("Source")["Price"].mean())
    # Visualizations 
    sns.set_theme(style="whitegrid")
    
    # 1. Bar Chart: Average Price Comparison
    plt.figure(figsize=(8, 6))
    sns.barplot(data=df, x="Source", y="Price", palette="viridis")
    plt.title(f"Average Price Comparison: {QUERY}")
    plt.ylabel("Price (INR)")
    plt.show()
    
    # 2. Scatter Plot: Price vs Rating
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x="Price", y="Rating", hue="Source", s=100, palette="deep")
    plt.title(f"Price vs Rating Distribution: {QUERY}")
    plt.grid(True)
    plt.show()

else:
    print("Error: No data retrieved from either source. Please check your internet connection or try again later.")

Starting Scrapers...
Processing Amazon for: HP Pavilion Laptop...
Amazon Extracted: 10 items
Processing Flipkart for: HP Pavilion Laptop...
Flipkart Extracted: 0 items


PermissionError: [Errno 13] Permission denied: 'laptop_price_comparison.csv'

Short Explanation of Scraping Logic

The program sends HTTP requests to the search pages of e-commerce websites (Amazon and Flipkart) for a selected laptop model. The returned HTML content is parsed using BeautifulSoup to locate product blocks. From each product listing, the script extracts the product name, price, and rating using HTML tags and class selectors. Missing values are handled safely to avoid errors. The extracted data from all websites is combined into a Pandas DataFrame, exported as a CSV file, and used to compute average prices for visual comparison using a bar chart.